# Notebook 06 — First-Order ODEs and Separable Equations

**Module:** 02 — Mathematics for Biology  
**Notebook:** 06 of 12  
**Prerequisites:** NB04  
**Time estimate:** 90 minutes

---
## Step 1 — Motivation

The exponential and logistic ODEs (NB04) are both *separable* — a class of first-order
ODEs you can solve analytically by hand in a few lines. Analytical solutions are
powerful: they tell you the long-term behaviour, the fixed points, and how the
solution changes when parameters change — without running any simulation. Knowing
when an ODE is separable and how to solve it is a core skill in mathematical biology.

---
## Step 2 — Intuition

A separable ODE has the form $dy/dx = g(x) \cdot h(y)$ — the right side factors into
a part that depends only on $x$ and a part that depends only on $y$. To solve it:
move all $y$ terms to one side, all $x$ terms to the other, then integrate both sides.

The key insight: you are asking 'what function $y(x)$ satisfies this relationship
between the function and its derivative?'

---
## Step 3 — Biological Background

**Three biological ODEs that are separable:**

| ODE | Biology | Separated form |
|-----|---------|----------------|
| $dN/dt = rN$ | Exponential growth | $dN/N = r\,dt$ |
| $dN/dt = rN(1-N/K)$ | Logistic growth | $dN / [N(1-N/K)] = r\,dt$ |
| $dC/dt = -k_e C$ | First-order drug clearance | $dC/C = -k_e\,dt$ |

**Fixed-point analysis (qualitative):**
Before solving, sketch $dN/dt$ vs $N$ (a *phase line*):
- Where $dN/dt > 0$: population increases
- Where $dN/dt < 0$: population decreases
- Where $dN/dt = 0$: fixed point

Arrows pointing toward a fixed point → stable equilibrium.
Arrows pointing away → unstable.

---
## Step 4 — Mathematical Explanation

**Separation of variables — general algorithm:**

Given $\frac{dy}{dx} = g(x) \cdot h(y)$:

1. Separate: $\frac{dy}{h(y)} = g(x)\,dx$
2. Integrate both sides: $\int \frac{dy}{h(y)} = \int g(x)\,dx + C$
3. Solve for $y(x)$ and apply initial condition to find $C$.

**Example: exponential growth**
$$\frac{dN}{N} = r\,dt \implies \int \frac{dN}{N} = \int r\,dt \implies \ln N = rt + C$$
$$\implies N(t) = e^C e^{rt} = N_0 e^{rt}$$

**Example: logistic growth (using partial fractions)**
$$\frac{dN}{N(1 - N/K)} = r\,dt$$
$$\frac{1}{N(1-N/K)} = \frac{1}{N} + \frac{1/K}{1 - N/K}$$
Integrating: $\ln N - \ln(1 - N/K) = rt + C$
Solving: $N(t) = \frac{K}{1 + (K/N_0 - 1)e^{-rt}}$

---
## Step 5 — Computational Explanation

**Phase line (1D) analysis in Python:**
1. Define $f(N) = dN/dt$ as a function
2. Evaluate it over a grid of $N$ values
3. Find zeros (fixed points): `scipy.optimize.brentq`
4. Determine sign of $f'(N^*)$ to classify stability

**Verify analytical solution vs. `solve_ivp`:**
Always compare your closed-form solution to a numerical ODE solver as a sanity check.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — Phase line analysis for logistic growth
r, K = 0.5, 100.0

def f_logistic(N: np.ndarray) -> np.ndarray:
    """dN/dt for logistic growth."""
    return r * N * (1 - N / K)

N_grid = np.linspace(0, 150, 300)
dN = f_logistic(N_grid)

# Find fixed points: where f(N) = 0
N_fp0 = 0.0
N_fp1 = brentq(f_logistic, 1, K + 50)  # search between 1 and K+50
print(f"Fixed points: N* = {N_fp0:.1f} and N* = {N_fp1:.1f}")

# Stability via sign of f near each fixed point
eps = 0.01
print(f"\nStability of N*=0:")
print(f"  f(0+ε) = {f_logistic(eps):.4f}  → arrows point {'away (unstable)' if f_logistic(eps) > 0 else 'toward (stable)'}")
print(f"\nStability of N*={N_fp1:.0f}:")
print(f"  f(K-ε) = {f_logistic(K - eps):.4f}  → arrows point {'toward (stable)' if f_logistic(K - eps) > 0 else 'away (unstable)'}")

In [ ]:
# Cell 6.2 — Analytical vs. numerical: logistic growth
def logistic_analytical(t: np.ndarray, r: float, K: float, N0: float) -> np.ndarray:
    """Closed-form solution to the logistic ODE."""
    return K / (1 + (K/N0 - 1) * np.exp(-r * t))

N0 = 5.0
t_span = (0, 30)
t_eval = np.linspace(0, 30, 300)

# Numerical solution via solve_ivp
sol = solve_ivp(lambda t, N: f_logistic(N), t_span, [N0], t_eval=t_eval, rtol=1e-8)
N_numerical = sol.y[0]

# Analytical solution
N_analytical = logistic_analytical(t_eval, r, K, N0)

max_error = np.max(np.abs(N_numerical - N_analytical))
print(f"Max absolute error (numerical vs. analytical): {max_error:.2e}")
print("(Should be < 1e-5 — ODE solver is highly accurate)")

In [ ]:
# Cell 6.3 — Multiple initial conditions: all converge to K
N0_vals = [1, 10, 50, 80, 120, 150]  # note: 120 and 150 are above K
t_ev = np.linspace(0, 25, 200)

trajectories = {N0_: logistic_analytical(t_ev, r, K, N0_) for N0_ in N0_vals}

print("All trajectories at t=25 (should all be near K=100):")
for N0_, traj in trajectories.items():
    print(f"  N0={N0_:<5} → N(25) = {traj[-1]:.2f}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Phase line and trajectories
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panel 1: Phase line — dN/dt vs N
ax = axes[0]
ax.plot(N_grid, dN, color="steelblue", lw=2)
ax.axhline(0, color="black", lw=0.8)
ax.axvline(0, color="black", lw=0.8)
ax.scatter([N_fp0, N_fp1], [0, 0], s=80, color=["tomato", "green"],
           zorder=5, label=["Unstable (N*=0)", f"Stable (N*={N_fp1:.0f})"])
# Add arrows
for N_arr in [20, 70, 110, 130]:
    direction = 1 if f_logistic(N_arr) > 0 else -1
    ax.annotate("", xy=(N_arr + 8*direction, f_logistic(N_arr)/15),
                xytext=(N_arr, f_logistic(N_arr)/15),
                arrowprops=dict(arrowstyle="->", color="gray"))
ax.set_xlabel("N"); ax.set_ylabel("dN/dt")
ax.set_title("Phase line — logistic growth")
ax.legend(handles=[
    plt.scatter([], [], color="tomato", s=80, label="Unstable (N*=0)"),
    plt.scatter([], [], color="green", s=80, label=f"Stable (N*={K:.0f})")
])

# Panel 2: Multiple trajectories
colors_traj = plt.cm.coolwarm(np.linspace(0, 1, len(N0_vals)))
ax = axes[1]
for (N0_, traj), color in zip(trajectories.items(), colors_traj):
    ax.plot(t_ev, traj, color=color, lw=1.5, label=f"N0={N0_}")
ax.axhline(K, color="black", ls="--", lw=1, label=f"K={K:.0f}")
ax.set_xlabel("t"); ax.set_ylabel("N(t)")
ax.set_title("All initial conditions converge to K")
ax.legend(fontsize=8, loc="right")

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Solve $dC/dt = -k_e C$ analytically using separation of variables. Verify your
   solution against `solve_ivp` for $C_0 = 100$, $k_e = 0.3$, $t \in [0, 20]$.
2. Draw the phase line (by hand) for $dN/dt = N(2 - N)(N - 0.5)$. Identify all fixed
   points and classify each as stable or unstable. Verify by computing trajectories
   from $N_0 = 0.1, 0.7, 1.5, 3.0$.
3. For the logistic ODE with $r = 0.5, K = 100$: what initial condition $N_0$ minimises
   the time to reach $N = 50$? Argue qualitatively from the phase line before computing.
4. Is the equation $dN/dt = t \cdot N^2$ separable? If so, solve it analytically.

---
## Quiz — Active Recall

1. What does it mean for an ODE to be 'separable'? Give one biological example.
2. What is a phase line? What do arrows on it represent?
3. How do you determine whether a fixed point is stable from the phase line?
4. Why does $N(t) \to K$ for all positive initial conditions in logistic growth?
5. Solve $dN/dt = rN$ analytically in three lines. Show your work.

---
## Reflection

**Date completed:** ____________________

> *[Can you do a phase-line analysis from scratch on a new ODE you've never seen? What step is hardest?]*

---
*Next: `07_euler_method.ipynb`*